In [2]:
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

pl.Config.set_tbl_rows(10)
pl.Config.set_tbl_cols(20)
pl.Config.set_fmt_str_lengths(150)

polars.config.Config

In [3]:
def cargar_y_unir_sentimientos_temas(nrc_scores_path: str,topics_dir: str,categories: list[str]):
    """
    Carga las puntuaciones de sentimiento y los resultados de modelado de temas,
    luego los une por categoría.

    Args:
        nrc_scores_path (str): Ruta al CSV con las puntuaciones de sentimiento NRC
                               (debe incluir: review_id, vader_score, afinn_score,
                               nrc_positive, nrc_negative).
        topics_dir (str):      Directorio con los CSVs de temas por categoría,
                               nombrados 'yelp_topics_<categoría>.csv' (debe incluir:
                               review_id, text, bertopic_dominant_topic).
        categories (list):     Lista de nombres de categorías a procesar (p. ej.
                               ["burgers", "fast_food"]).

    """
    topic_files = {cat: os.path.join(topics_dir, f"yelp_topics_{cat}.csv") for cat in categories}

    scores_lazy = (
        pl.scan_csv(nrc_scores_path)
        .select(['review_id', 'vader_score', 'afinn_score', 'nrc_positive', 'nrc_negative'])
        .with_columns([
            (pl.col("vader_score") - pl.col("afinn_score")).abs().alias("lexicon_disagreement")
        ])
    )

    category_dfs = {}
    for cat in categories:
        print(f"Procesando categoría: {cat}...")

        topic_lazy = (
            pl.scan_csv(topic_files[cat])
            .select(['review_id', 'text', 'bertopic_dominant_topic'])
        )

        joined_df = (
            topic_lazy
            .join(scores_lazy, on="review_id", how="left")
            .collect()
        )

        category_dfs[cat] = joined_df
        print(f"  -> Total procesado para {cat}: {joined_df.height} reviews.")

    return category_dfs


nrc_scores_path = "../results/sentiment_analysis/yelp_academic_dataset_review_scored.csv"
topics_dir = "../results/topic_modeling/"
categories = ["burgers", "fast_food", "mexican", "pubs", "steakhouses"]

category_dfs = cargar_y_unir_sentimientos_temas(nrc_scores_path, topics_dir, categories)

Procesando categoría: burgers...
  -> Total procesado para burgers: 445895 reviews.
Procesando categoría: fast_food...
  -> Total procesado para fast_food: 233008 reviews.
Procesando categoría: mexican...
  -> Total procesado para mexican: 432248 reviews.
Procesando categoría: pubs...
  -> Total procesado para pubs: 218891 reviews.
Procesando categoría: steakhouses...
  -> Total procesado para steakhouses: 240040 reviews.


In [4]:
def mostrar_desacuerdo_por_topico(category_dfs: dict[str, pl.DataFrame],categories: list[str],min_reviews: int = 50,top_n: int = 5):
    """
    Muestra los tópicos con mayor desacuerdo léxico medio por categoría.

    Args:
        category_dfs (dict):  Diccionario categoría -> DataFrame con columnas bertopic_dominant_topic y lexicon_disagreement.
        categories (list):    Lista de categorías a analizar.
        min_reviews (int):    Mínimo de reseñas por tópico para incluirlo (default: 50).
        top_n (int):          Número de tópicos a mostrar por categoría (default: 5).
    """
    for cat in categories:
        df = category_dfs[cat].drop_nulls(
            subset=["bertopic_dominant_topic", "lexicon_disagreement"]
        )

        topic_disagreement = (
            df.group_by("bertopic_dominant_topic")
            .agg([
                pl.col("lexicon_disagreement").mean().alias("mean_disagreement"),
                pl.len().alias("num_reviews")
            ])
            .filter(pl.col("num_reviews") > min_reviews)
            .sort("mean_disagreement", descending=True)
        )

        print(f"\n[{cat.capitalize()}] Top {top_n} tópicos con mayor desacuerdo léxico:")
        display(topic_disagreement.head(top_n))


def mostrar_reviews_mayor_desacuerdo(category_dfs: dict[str, pl.DataFrame],topic_files: dict[str, str],categories: list[str],top_n: int = 20):
    """
    Muestra las reseñas con mayor desacuerdo léxico por categoría, indicando qué lexicón se aproxima más a la puntuación real de estrellas.

    Args:
        category_dfs (dict):  Diccionario categoría -> DataFrame con columnaslexicon_disagreement, vader_score y afinn_score.
        topic_files (dict):   Diccionario categoría -> ruta al CSV de temas, usado para obtener la columna stars.
        categories (list):    Lista de categorías a analizar.
        top_n (int):          Número de reseñas a mostrar por categoría (default: 20).
    """
    for cat in categories:
        df = category_dfs[cat].drop_nulls(subset=["lexicon_disagreement"])

        stars_lazy = pl.scan_csv(topic_files[cat]).select(["review_id", "stars"])
        top_ambiguous = (
            pl.LazyFrame(df)
            .join(stars_lazy, on="review_id", how="left")
            .sort("lexicon_disagreement", descending=True)
            .head(top_n)
            .collect()
        )

        print(f"\n{'='*80}")
        print(f"Top {top_n} reviews con MAYOR DESACUERDO — {cat.upper()}")
        print(f"{'='*80}")

        for row in top_ambiguous.iter_rows(named=True):
            vader_correct = (
                abs(row['vader_score'] - row['stars']) < abs(row['afinn_score'] - row['stars'])
            )

            print(f"\nReview ID  : {row['review_id']}")
            print(f"Tópico     : {row['bertopic_dominant_topic']}")
            print(f"Desacuerdo : {row['lexicon_disagreement']:.4f}")
            print(f"Estrellas reales : {row['stars']:.1f}")
            print(f"Scores → VADER: {row['vader_score']:.3f} | AFINN: {row['afinn_score']:.3f}")
            print(f"Más cercano a stars: {'VADER ' if vader_correct else 'AFINN '}")
            print(f"Texto      : {row['text'].replace(chr(10), ' ')[:400]}...")
            print("-" * 80)


topic_files = {cat: os.path.join(topics_dir, f"yelp_topics_{cat}.csv") for cat in categories}

mostrar_desacuerdo_por_topico(category_dfs, categories)
mostrar_reviews_mayor_desacuerdo(category_dfs, topic_files, categories)


[Burgers] Top 5 tópicos con mayor desacuerdo léxico:


bertopic_dominant_topic,mean_disagreement,num_reviews
str,f64,u32
"""202_use coupon_coupon free_tried use_free burger""",0.929057,101
"""195_dogs allowed_allow dogs_dog friendly_health code""",0.819823,150
"""245_18 gratuity_20 gratuity_tip 20_party 20""",0.793225,109
"""432_iced coffee_cream sugar_ice coffee_coffee didnt""",0.771614,52
"""233_sure hype_dont hype_know hype_hype burger""",0.766233,126



[Fast_food] Top 5 tópicos con mayor desacuerdo léxico:


bertopic_dominant_topic,mean_disagreement,num_reviews
str,f64,u32
"""278_accept coupons_dont accept_use coupon_foot long""",1.088026,51
"""132_use coupon_tried use_accept coupons_expiration date""",0.913693,149
"""56_credit card_gift card_debit card_gift cards""",0.877174,364
"""63_pita pit_chicken pita_chicken souvlaki_tzatziki sauce""",0.802429,339
"""284_kiosk order_order kiosk_told use_royal farms""",0.745818,51



[Mexican] Top 5 tópicos con mayor desacuerdo léxico:


...


[Pubs] Top 5 tópicos con mayor desacuerdo léxico:


bertopic_dominant_topic,mean_disagreement,num_reviews
str,f64,u32
"""8_minutes later_10 minutes_15 minutes_20 minutes""",0.795098,1149
"""62_told leave_extremely rude_rude bartender_african american""",0.788845,199
"""61_years old_asked id_wouldnt let_turned away""",0.787626,233
"""163_tables dirty_deep cleaning_paper towels_table dirty""",0.762971,86
"""312_fruit flies_fruit fly_flies buzzing_think point""",0.7616,52



[Steakhouses] Top 5 tópicos con mayor desacuerdo léxico:


bertopic_dominant_topic,mean_disagreement,num_reviews
str,f64,u32
"""229_credit card_money pay_debit card_bank account""",1.081592,60
"""267_ahead seating_calling ahead_called ahead_ahead waiting""",0.927691,53
"""43_minutes later_walked away_ready order_10 minutes""",0.862623,275
"""199_sick eating_stomach ache_hours eating_feel sick""",0.798456,74
"""185_health department_looked saw_told waitress_paper towels""",0.780551,98



Top 20 reviews con MAYOR DESACUERDO — BURGERS

Review ID  : gDtOEIh9whrbotVqJHdeUg
Tópico     : -1_mac cheese_sweet potato_onion rings_potato fries
Desacuerdo : 3.9568
Estrellas reales : 2.0
Scores → VADER: 1.036 | AFINN: 4.993
Más cercano a stars: VADER 
Texto      : I just want to start by saying that our server, Will, was wonderful. He was helpful and attentive. If we didn't like an item he took it back to get us something he thought we would like better. The restaurant is also along the river so the atmosphere is nice. If I was rating Will and the location it would be 5 stars...but I'm not.   This restaurant has poor quality food and it doesn't taste good a...
--------------------------------------------------------------------------------

Review ID  : lDX6ryaTLoCUH8XGsDu0zQ
Tópico     : -1_mac cheese_sweet potato_onion rings_potato fries
Desacuerdo : 3.9446
Estrellas reales : 2.0
Scores → VADER: 1.022 | AFINN: 4.967
Más cercano a stars: VADER 
Texto      : This restaurant had a 